<a href="https://colab.research.google.com/github/gbyellewfr/KKA-PraktikumEDA/blob/main/KelompokProjectCharter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D. Lembar Perencanaan Proyek Kelompok (Project Charter)

## 1. Nama Kelompok
- Anggota 1: Nama Anggota 1
- Anggota 2: Nama Anggota 2

## 2. Dataset yang Dipilih
Dataset yang digunakan adalah dataset penjualan kantin dengan nama file:
`dataset_penjualan_kantin.csv`

Dataset berisi data transaksi penjualan kantin, seperti:
- ID transaksi
- tanggal transaksi
- nama produk
- kategori produk
- jumlah produk terjual
- harga satuan
- nama kasir
- metode pembayaran

## 3. Pertanyaan Analisis Awal
1. Produk apa yang memiliki jumlah penjualan paling banyak?
2. Kategori produk apa yang menghasilkan total pendapatan terbesar?
3. Metode pembayaran apa yang paling sering digunakan pelanggan?

## 4. Dugaan Masalah Kualitas Data
Beberapa masalah yang ditemukan:
- Terdapat nilai kosong pada `jumlah_terjual`, `harga_satuan`, dan `nama_kasir`.
- Terdapat data duplikat.
- Tipe data beberapa kolom masih berupa object/string.
- Penulisan kategori tidak konsisten, misalnya `Makanan`, `makanan`, dan `MAKANAN`.
- Beberapa jumlah terjual memiliki tulisan `pcs`.
- Harga memiliki format seperti `Rp2.000`.
- Terdapat nilai jumlah terjual yang tidak wajar, yaitu 500.

## 5. Rencana Cleaning
- Menghapus data duplikat.
- Mengubah tanggal menjadi tipe datetime.
- Mengubah `jumlah_terjual` menjadi numerik.
- Menghapus teks `pcs` dari jumlah terjual.
- Mengubah `harga_satuan` menjadi numerik.
- Menyeragamkan penulisan kategori.
- Missing value `jumlah_terjual` diisi berdasarkan median jumlah penjualan produk yang sama.
- Missing value `harga_satuan` diisi berdasarkan median harga produk yang sama.
- Missing value `nama_kasir` diisi dengan `Tidak diketahui`.
- Nilai jumlah terjual yang tidak wajar akan ditandai dan diperlakukan sebagai missing sebelum dilakukan pengisian.

## 6. Rencana Manipulasi Data
- Filtering transaksi dengan total penjualan tertentu.
- Sorting produk berdasarkan jumlah terjual.
- Membuat kolom turunan `total_penjualan`.
- Melakukan groupby berdasarkan produk dan kategori.
- Menghitung total jumlah terjual dan total pendapatan.

## 7. Jadwal Pengerjaan
| Pertemuan | Kegiatan |
|---|---|
| P3 | Menentukan dataset dan pertanyaan analisis |
| P4 | Data loading, inspection, dan cleaning |
| P5 | Data manipulation dan analisis |
| P6 | Membuat kesimpulan dan presentasi |

## 8. Pembagian Tugas
| Tugas | Penanggung Jawab |
|---|---|
| Menyiapkan dataset | Anggota 1 |
| Data cleaning | Anggota 1 |
| Data manipulation | Anggota 2 |
| Analisis hasil | Anggota 2 |
| Kesimpulan dan presentasi | Semua anggota |

# E. Proyek Akhir — EDA Starter Project

## Analisis Data Penjualan Kantin

Tujuan analisis ini adalah memahami pola penjualan kantin berdasarkan produk, kategori, jumlah penjualan, pendapatan, dan metode pembayaran.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
df = pd.read_csv('dataset_penjualan_kantin.csv')

df.head()


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# Data Cleaning

Pada tahap ini dilakukan:
1. Menghapus data duplikat.
2. Membersihkan format jumlah terjual.
3. Membersihkan format harga.
4. Menangani nilai yang tidak wajar.
5. Menangani missing value.
6. Menyeragamkan kategori.
7. Mengubah tipe data tanggal.

In [ ]:
df = df.drop_duplicates()

print("Jumlah data setelah menghapus duplikat:", len(df))

In [ ]:
df['jumlah_terjual'] = (
    df['jumlah_terjual']
    .astype(str)
    .str.extract(r'(\d+(?:\.\d+)?)')[0]
)

df['jumlah_terjual'] = pd.to_numeric(
    df['jumlah_terjual'],
    errors='coerce'
)

df.head()

In [ ]:
# Nilai jumlah terjual yang terlalu besar dianggap tidak wajar
df.loc[df['jumlah_terjual'] > 20, 'jumlah_terjual'] = np.nan

df['jumlah_terjual'].describe()

In [ ]:
df['harga_satuan'] = (
    df['harga_satuan']
    .astype(str)
    .str.replace(r'[^\d]', '', regex=True)
)

df['harga_satuan'] = pd.to_numeric(
    df['harga_satuan'],
    errors='coerce'
)

df.head()

In [ ]:

df['kategori'] = df['kategori'].str.strip().str.title()

df['kategori'].value_counts()

In [ ]:
bulan_indonesia = {
    'Januari': 'January',
    'Februari': 'February',
    'Maret': 'March',
    'April': 'April',
    'Mei': 'May',
    'Juni': 'June',
    'Juli': 'July',
    'Agustus': 'August',
    'September': 'September',
    'Oktober': 'October',
    'November': 'November',
    'Desember': 'December'
}

for indo, english in bulan_indonesia.items():
    df['tanggal'] = df['tanggal'].str.replace(
        indo,
        english,
        regex=False
    )

df['tanggal'] = pd.to_datetime(
    df['tanggal'],
    dayfirst=True,
    errors='coerce'
)

df['tanggal'].head()

In [ ]:
df.isnull().sum()

In [ ]:

df['jumlah_terjual'] = (
    df['jumlah_terjual']
    .fillna(
        df.groupby('nama_produk')['jumlah_terjual']
        .transform('median')
    )
)

df['jumlah_terjual'] = df['jumlah_terjual'].fillna(
    df['jumlah_terjual'].median()
)

In [ ]:
df['harga_satuan'] = (
    df['harga_satuan']
    .fillna(
        df.groupby('nama_produk')['harga_satuan']
        .transform('median')
    )
)

df['harga_satuan'] = df['harga_satuan'].fillna(
    df['harga_satuan'].median()
)

In [ ]:
df['nama_kasir'] = df['nama_kasir'].fillna('Tidak diketahui')

In [ ]:
df['id_transaksi'] = df['id_transaksi'].astype(str)
df['nama_produk'] = df['nama_produk'].astype(str)
df['kategori'] = df['kategori'].astype(str)
df['nama_kasir'] = df['nama_kasir'].astype(str)
df['metode_pembayaran'] = df['metode_pembayaran'].astype(str)

df['jumlah_terjual'] = df['jumlah_terjual'].astype(int)
df['harga_satuan'] = df['harga_satuan'].astype(int)
df['tanggal'] = pd.to_datetime(df['tanggal'])

df.info()

In [ ]:
print("Missing value:")
print(df.isnull().sum())

print("\nJumlah duplikat:")
print(df.duplicated().sum())

print("\nShape:")
print(df.shape)

# Data Manipulation

Pada tahap ini dilakukan:
- Filtering
- Sorting
- Membuat kolom turunan
- Groupby dan agregasi

In [ ]:
df['total_penjualan'] = (
    df['jumlah_terjual'] * df['harga_satuan']
)

df.head()

In [ ]:
transaksi_besar = df[df['total_penjualan'] > 50000]

transaksi_besar.head(10)

In [ ]:
produk_terlaris = df.sort_values(
    by='jumlah_terjual',
    ascending=False
)

produk_terlaris.head(10)

In [ ]:
penjualan_produk = (
    df.groupby('nama_produk')
    .agg(
        total_terjual=('jumlah_terjual', 'sum'),
        total_pendapatan=('total_penjualan', 'sum')
    )
    .sort_values(
        by='total_terjual',
        ascending=False
    )
)

penjualan_produk

In [ ]:
penjualan_kategori = (
    df.groupby('kategori')
    .agg(
        total_terjual=('jumlah_terjual', 'sum'),
        total_pendapatan=('total_penjualan', 'sum')
    )
    .sort_values(
        by='total_pendapatan',
        ascending=False
    )
)

penjualan_kategori

In [ ]:
pembayaran = (
    df.groupby('metode_pembayaran')
    .agg(
        jumlah_transaksi=('id_transaksi', 'count'),
        total_pendapatan=('total_penjualan', 'sum')
    )
    .sort_values(
        by='jumlah_transaksi',
        ascending=False
    )
)

pembayaran

In [ ]:
penjualan_produk['total_terjual'].head(10).plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('10 Produk dengan Jumlah Terjual Terbanyak')
plt.xlabel('Nama Produk')
plt.ylabel('Total Terjual')
plt.xticks(rotation=45)
plt.show()

In [ ]:
penjualan_kategori['total_pendapatan'].plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('Total Pendapatan Berdasarkan Kategori')
plt.xlabel('Kategori')
plt.ylabel('Total Pendapatan')
plt.xticks(rotation=0)
plt.show()

In [ ]:
pembayaran['jumlah_transaksi'].plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('Jumlah Transaksi Berdasarkan Metode Pembayaran')
plt.xlabel('Metode Pembayaran')
plt.ylabel('Jumlah Transaksi')
plt.xticks(rotation=0)
plt.show()

In [ ]:
print("=== PRODUK TERLARIS ===")
print(penjualan_produk.head(5))

print("\n=== KATEGORI DENGAN PENDAPATAN TERBESAR ===")
print(penjualan_kategori.head(5))

print("\n=== METODE PEMBAYARAN ===")
print(pembayaran)

# Data Profiling Summary

Berdasarkan hasil eksplorasi data penjualan kantin, ditemukan beberapa pola utama.

1. Produk dengan total jumlah terjual paling tinggi adalah **Nasi Goreng**. Hal ini menunjukkan bahwa produk tersebut memiliki volume penjualan yang tinggi dibandingkan produk lainnya.

2. Berdasarkan total pendapatan, kategori **Makanan** memberikan kontribusi pendapatan paling besar dibandingkan kategori Minuman dan Snack.

3. **QRIS** merupakan metode pembayaran yang paling banyak digunakan dalam transaksi pada dataset setelah data dibersihkan.

Selain itu, proses cleaning menunjukkan bahwa dataset awal memiliki beberapa masalah kualitas data seperti missing value, data duplikat, perbedaan format penulisan kategori, format harga yang berbeda, serta format jumlah terjual yang masih mengandung teks seperti `pcs`.

In [ ]:
df.to_csv(
    'dataset_penjualan_kantin_bersih.csv',
    index=False
)

print("Dataset bersih berhasil disimpan.")

In [ ]:
from google.colab import files

files.download('dataset_penjualan_kantin_bersih.csv')

In [ ]:
print("===== CEK AKHIR DATASET =====")

print("Jumlah baris dan kolom:", df.shape)

print("\nMissing value:")
print(df.isnull().sum())

print("\nJumlah duplikat:", df.duplicated().sum())

print("\nTipe data:")
print(df.dtypes)

print("\n5 data pertama:")
display(df.head())